In [27]:
import matplotlib.pyplot as plt # For plotting
import numpy as np              # Linear algebra library
import pandas as pd
import sys
import csv
from collections import Counter

In [28]:
def make_bow(data, vocab):
    X = np.zeros([len(data), len(vocab)], dtype=int)
    t = []

    word_index = {word: idx for idx, word in enumerate(vocab)}  # fast lookup

    for i, (review, label) in enumerate(data):
        words = set(review.split())
        for word in words:
            if word in word_index:
                X[i, word_index[word]] = 1
        t.append(label)

    return X, np.array(t)

In [29]:
def word_to_digit(sentence):
    word_to_number = {
        "one": "1",
        "two": "2",
        "three": "3",
        "four": "4",
        "five": "5",
        "six": "6",
        "seven": "7",
        "eight": "8",
        "nine": "9",
        "zero": "0",
        "ten": "10",
        "eleven": "11",
        "twelve": "12",
        "thirteen": "13",
        "fourteen": "14",
        "fifteen": "15",
        "sixteen": "16",
        "seventeen": "17",
        "eighteen": "18",
        "nineteen": "19",
        "twenty": "20"
        # Add more words as needed
    }
    for key in word_to_number:
        if key in sentence:
            return word_to_number[key]
    return sentence

In [30]:
import re

text = "about four"

def first_number(text):
    if pd.isna(text):
        return np.nan

    text = str(text).lower()
    text = word_to_digit(text)

    # Try to extract digits first
    match = re.search(r'\d+', text)
    if match:
        return int(match.group())
    else:
      return None

first_number(text)

4

In [31]:
def categorize_drink(drink_text):
    if pd.isna(drink_text):
        return 'none'

    drink_text = drink_text.lower().strip()

    # Specific drinks
    if 'sake' in drink_text or 'soju' in drink_text:
        return 'asian alcohol'
    elif 'tea' in drink_text:
        return 'tea'
    elif 'ayran' in drink_text or 'lassi' in drink_text or 'yogurt' in drink_text:
        return 'yogurt drink'
    # elif 'coffee' in drink_text or 'espresso' in drink_text:
    #     return 'coffee'
    elif 'coke' in drink_text or 'pepsi' in drink_text or 'cola' in drink_text or 'soda' in drink_text or 'root beer' in drink_text or 'sprite' in drink_text or 'pop' in drink_text or 'ginger ale' in drink_text or 'soft drink' in drink_text or 'fanta' in drink_text:
        return 'soda'
    elif 'wine' in drink_text or 'beer' in drink_text or 'cocktail' in drink_text or 'margarita' in drink_text or 'mojito' in drink_text or 'alcohol' in drink_text:
        return 'other alcohol'
    elif 'juice' in drink_text:
        return 'juice'
    elif 'water' in drink_text:
        return 'water'
    elif 'soup' in drink_text:
        return 'soup'
    elif "don't drink" in drink_text or 'none' in drink_text or 'no drink' in drink_text:
        return 'none'
    else:
        return 'other'


categorize_drink("WaterAAAA")

'water'

In [32]:
def preprocess_data(data):
  # read each of the csv files as a *pandas data frame*
  # data = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/cleaned_data_combined.csv")

  q2 = 'Q2: How many ingredients would you expect this food item to contain?'
  q4 = 'Q4: How much would you expect to pay for one serving of this food item?'
  q6 = 'Q6: What drink would you pair with this food item?'
  q8 = 'Q8: How much hot sauce would you add to this food item?'
  data[q8] = data[q8].replace(pd.NA, 0)
  data[q8] = data[q8].replace('A little (mild)', 1)
  data[q8] = data[q8].replace('A moderate amount (medium)', 2)
  data[q8] = data[q8].replace('A lot (hot)', 3)
  data[q8] = data[q8].replace('I will have some of this food item with my hot sauce', 4)

  # replace q2 and q4 data columns with the first number that appears
  data[q2] = data[q2].apply(first_number)
  data[q4] = data[q4].apply(first_number)

  # categorize drinks into usable data
  data[q6] = data[q6].apply(categorize_drink)

  # fill NaN with mean
  data[q2] = data[q2].fillna(
      data.groupby('Label')[q2].transform('mean')
  )

  data[q4] = data[q4].fillna(
      data.groupby('Label')[q4].transform('mean')
  )

  # convert labels to numbers for Logistic Regression
  data['Label'] = data['Label'].replace('Pizza', 0)
  data['Label'] = data['Label'].replace('Shawarma', 1)
  data['Label'] = data['Label'].replace('Sushi', 2)

  movie_reviews = data["Q5: What movie do you think of when thinking of this food item?"].fillna("").astype(str).str.lower()
  food_labels = data["Label"]
  movie_data = list(zip(movie_reviews, food_labels))

  # Build vocabulary (simple token split, no stopwords removal or punctuation filtering for now)
  tokenized_reviews = [set(review.split()) for review, _ in movie_data]
  vocab_set = set(word for tokens in tokenized_reviews for word in tokens)
  vocab = sorted(vocab_set)

  X_bow, t_labels = make_bow(movie_data, vocab)

  # display one the dataframes in the notebook
  return data, X_bow

In [33]:
food_complexity = "Q1: From a scale 1 to 5, how complex is it to make this food? (Where 1 is the most simple, and 5 is the most complex)"

setting = "Q3: In what setting would you expect this food to be served? Please check all that apply"
drinks = "Q6: What drink would you pair with this food item?"
person = "Q7: When you think about this food item, who does it remind you of?"
hot_sauce = "Q8: How much hot sauce would you add to this food item?"
def encode_data(data, X_bow):
  data_fets = np.stack([
      # 5 choices for food complexity
      data[food_complexity] == 1,
      data[food_complexity] == 2,
      data[food_complexity] == 3,
      data[food_complexity] == 4,
      data[food_complexity] == 5,

      # 5 choices for hot sauce
      data[hot_sauce] == 0,
      data[hot_sauce] == 1,
      data[hot_sauce] == 2,
      data[hot_sauce] == 3,
      data[hot_sauce] == 4,

      # 7 categories, for each type of setting including None
      data[setting].str.contains("Week day lunch", na=False).astype(int),
      data[setting].str.contains("Week day dinner", na=False).astype(int),
      data[setting].str.contains("Weekend lunch", na=False).astype(int),
      data[setting].str.contains("Weekend dinner", na=False).astype(int),
      data[setting].str.contains("At a party", na=False).astype(int),
      data[setting].str.contains("Late night snack", na=False).astype(int),
      data[setting] == pd.NA,

      # 6 categories for each type of person and None
      data[person].str.contains("Parents", na=False).astype(int),
      data[person].str.contains("Siblings", na=False).astype(int),
      data[person].str.contains("Friends", na=False).astype(int),
      data[person].str.contains("Teachers", na=False).astype(int),
      data[person].str.contains("Strangers", na=False).astype(int),
      data[person] == pd.NA,

      # 10 categories for drinks
      data[drinks].str.contains("soda", na=False).astype(int),
      data[drinks].str.contains("water", na=False).astype(int),
      data[drinks].str.contains("tea", na=False).astype(int),
      data[drinks].str.contains("other", na=False).astype(int),
      data[drinks].str.contains("juice", na=False).astype(int),
      data[drinks].str.contains("asian alcohol", na=False).astype(int),
      data[drinks].str.contains("other alcohol", na=False).astype(int),
      data[drinks].str.contains("soup", na=False).astype(int),
      data[drinks].str.contains("yogurt drink", na=False).astype(int),
      data[drinks].str.contains("none", na=False).astype(int),

      # numerical features
      data["Q2: How many ingredients would you expect this food item to contain?"],
      data["Q4: How much would you expect to pay for one serving of this food item?"]
  ], axis=1)
  # Convert sparse BoW to dense (only if memory permits)
  # bow_dense = X_bow.toarray()

  # Sanity check: shapes should align on axis 0
  assert data_fets.shape[0] == X_bow.shape[0]

  # Concatenate along columns (axis=1)
  data_fets = np.concatenate([X_bow, data_fets], axis=1)

  # Confirm shape
  print("Combined feature shape:", data_fets.shape)

  return data_fets

# print(data_fets.shape) # Should be (1644, 35)

In [34]:
import json

with open("/content/drive/MyDrive/Colab Notebooks/random_forest_trees.json", "r") as f:
    forest_data = json.load(f)

def predict_tree(x, tree):
    node = 0
    while tree["children_left"][node] != tree["children_right"][node]:
        feature = tree["feature"][node]
        threshold = tree["threshold"][node]
        if x[feature] <= threshold:
            node = tree["children_left"][node]
        else:
            node = tree["children_right"][node]
    class_counts = tree["value"][node][0]
    return int(np.argmax(class_counts))

def predict_forest(x, forest_data):
    vote_counts = {}

    for tree in forest_data:
        pred = predict_tree(x, tree)
        if pred not in vote_counts:
            vote_counts[pred] = 1
        else:
            vote_counts[pred] += 1

    # Find the class with the highest vote count
    majority_class = max(vote_counts, key=vote_counts.get)
    return majority_class

In [35]:
"""
Here's an example of how your script may be used in our test file:

    from example_pred import predict_all
    predict_all("example_test_set.csv")
"""

def predict(x):
    """
    Helper function to make prediction for a given input x.
    This code is here for demonstration purposes only.
    """
    # randomly choose between the three choices: 'Pizza', 'Shawarma', 'Sushi'.
    # NOTE: make sure to be *very* careful of the spelling/capitalization of the cities!!
    y = predict_forest(x, forest_data)

    # return the prediction
    return y


def predict_all(filename):
    """
    Make predictions for the data in filename
    """
    label_map = {
    0: "Pizza",
    1: "Shawarma",
    2: "Sushi"
    }
    # read the file containing the test data
    # you do not need to use the "csv" package like we are using
    # (e.g. you may use numpy, pandas, etc)
    data = pd.read_csv(open(filename))
    data, X_bow = preprocess_data(data)
    # print(X_bow)
    data = encode_data(data, X_bow)
    # print(data.shape)
    predictions = []
    for row in data:
        # obtain a prediction for this test example
        # print(row)
        pred = predict(row)
        predictions.append(label_map[pred])

    return predictions

In [36]:
label_map = {
    0: "Pizza",
    1: "Shawarma",
    2: "Sushi"
    }
def accuracy(pred):
  data = pd.read_csv(open('./cleaned_data_combined.csv'))
  data, X_bow = preprocess_data(data)
  true = data['Label']
  true = [label_map[p] for p in true]
  correct = sum(p == t for p, t in zip(pred, true))
  return correct / len(true)

In [42]:
accuracy(predict_all('./cleaned_data_combined.csv'))

<ipython-input-32-a55b01fc825c>:13: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data[q8] = data[q8].replace('I will have some of this food item with my hot sauce', 4)
<ipython-input-32-a55b01fc825c>:34: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data['Label'] = data['Label'].replace('Sushi', 2)


Combined feature shape: (1644, 1222)


<ipython-input-32-a55b01fc825c>:13: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data[q8] = data[q8].replace('I will have some of this food item with my hot sauce', 4)
<ipython-input-32-a55b01fc825c>:34: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data['Label'] = data['Label'].replace('Sushi', 2)


0.9038929440389294